<a href="https://colab.research.google.com/github/2403A51L03/Reinforcement_Learning/blob/main/RL_Lab_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile grid_env_det.py
# grid_env_det.py
# Environment definition for 2x3 Grid World MDP

class GridWorldMDP:
    def __init__(self):
        # a. State Space (X is an obstacle and not a valid state)
        self.states = ["S0", "S1", "G", "S3", "S4"]

        # Coordinates for reference: S0=(0,0), S1=(0,1), G=(0,2), X=(1,0), S3=(1,1), S4=(1,2)
        self.state_coords = {
            "S0": (0, 0),
            "S1": (0, 1),
            "G": (0, 2),
            "S3": (1, 1),
            "S4": (1, 2)
        }

        # b. Action Space
        self.actions = ["Up", "Down", "Left", "Right"]

        # d. Discount Factor
        self.gamma = 0.9

        # Goal definition
        self.goal_state = "G"

        # e. Deterministic Transition Table P(s' | s, a)
        self.transition_table = {
            "S0": {"Up": "S0", "Down": "S0", "Left": "S0", "Right": "S1"},
            "S1": {"Up": "S1", "Down": "S3", "Left": "S0", "Right": "G"},
            "S3": {"Up": "S1", "Down": "S3", "Left": "S3", "Right": "S4"},
            "S4": {"Up": "G",  "Down": "S4", "Left": "S3", "Right": "S4"},
            "G":  {"Up": "G",  "Down": "G",  "Left": "G",  "Right": "G"}
        }

    # c. Define Reward Function
    def get_reward(self, current_state, action, next_state):
        if current_state == self.goal_state:
            return 0  # Episode terminated
        if next_state == self.goal_state:
            return 10  # Goal reward
        return -1  # Normal movement reward

    # f. Transition Function to estimate next state
    def transition(self, state, action):
        if state not in self.states:
            raise ValueError(f"State '{state}' is not a valid state.")
        if action not in self.actions:
            raise ValueError(f"Action '{action}' is not a valid action.")
        return self.transition_table[state][action]

Writing grid_env_det.py


In [2]:
# main.py
# Program to import grid_env_det and test transition functionality

from grid_env_det import GridWorldMDP


def main():
    # Instantiate the MDP Environment
    env = GridWorldMDP()

    print("==================================================")
    print("      2 x 3 GRID WORLD MDP ENVIRONMENT DEMO       ")
    print("==================================================")

    # a. Display State Space
    print(f"\na. State Space (S): {env.states}")

    # b. Display Action Space
    print(f"b. Action Space (A): {env.actions}")

    # d. Display Discount Factor
    print(f"d. Discount Factor (gamma): {env.gamma}")

    # Display Transition Matrix Table
    print("\n--------------------------------------------------")
    print("f. Full Transition Function & Reward Table:")
    print("--------------------------------------------------")
    print(f"{'Current State':<15} {'Action':<10} {'Next State':<12} {'Reward':<8}")
    print("-" * 50)

    for state in env.states:
        for action in env.actions:
            next_state = env.transition(state, action)
            reward = env.get_reward(state, action, next_state)
            print(
                f"{state:<15} {action:<10} {next_state:<12} {reward:<8}"
            )

    # f. Interactive Next State Estimation Test
    print("\n--------------------------------------------------")
    print("   Interactive Next State Estimation Test         ")
    print("--------------------------------------------------")

    sample_queries = [
        ("S0", "Right"),
        ("S0", "Down"),
        ("S1", "Right"),
        ("S3", "Left"),
        ("S4", "Up"),
    ]

    for curr_s, act in sample_queries:
        nxt_s = env.transition(curr_s, act)
        r = env.get_reward(curr_s, act, nxt_s)
        print(
            f"Given State = {curr_s:2s} | Action = {act:5s} ==> Next State = {nxt_s:2s} | Reward = {r:3d}"
        )


if __name__ == "__main__":
    main()

      2 x 3 GRID WORLD MDP ENVIRONMENT DEMO       

a. State Space (S): ['S0', 'S1', 'G', 'S3', 'S4']
b. Action Space (A): ['Up', 'Down', 'Left', 'Right']
d. Discount Factor (gamma): 0.9

--------------------------------------------------
f. Full Transition Function & Reward Table:
--------------------------------------------------
Current State   Action     Next State   Reward  
--------------------------------------------------
S0              Up         S0           -1      
S0              Down       S0           -1      
S0              Left       S0           -1      
S0              Right      S1           -1      
S1              Up         S1           -1      
S1              Down       S3           -1      
S1              Left       S0           -1      
S1              Right      G            10      
G               Up         G            0       
G               Down       G            0       
G               Left       G            0       
G               Right      

In [3]:
from grid_env_det import GridWorldMDP


# Policy
policy = {
    "S0": "Right",
    "S1": "Down",
    "S3": "Right",
    "S4": "Up"
}


def state_value_function(env, start_state):
    state = start_state
    rewards = []

    while state != env.goal_state:
        action = policy[state]
        next_state = env.transition(state, action)

        reward = env.get_reward(state, action, next_state)
        rewards.append(reward)

        state = next_state

    # Calculate discounted return
    value = 0

    for t in range(len(rewards)):
        value += (env.gamma ** t) * rewards[t]

    return value


def action_value_function(env, state, action):
    next_state = env.transition(state, action)
    reward = env.get_reward(state, action, next_state)

    # If goal is reached, episode terminates
    if next_state == env.goal_state:
        return reward

    # Follow policy from next state
    future_value = state_value_function(env, next_state)

    return reward + env.gamma * future_value


def main():
    env = GridWorldMDP()

    state_value = state_value_function(env, "S0")

    action_value = action_value_function(
        env,
        "S0",
        "Up"
    )

    print("State Value Function")
    print("--------------------")
    print("V(S0) =", state_value)

    print("\nAction Value Function")
    print("--------------------")
    print("Q(S0, Up) =", action_value)


if __name__ == "__main__":
    main()

State Value Function
--------------------
V(S0) = 4.580000000000001

Action Value Function
--------------------
Q(S0, Up) = 3.1220000000000008
